### 📈 Everything is technical analysis

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 🎲 A simple question not many can answer it coherently

What is the difference between technical analysis, drawing lines on a chart, and your backtest?

    In one, you draw a line on a chart and *assume* the trend will continue into the future.

    In the other, you come up with an *algorithm* that assumes the trend will continue into the future.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 15
N_BACKTEST_PATHS = 24
FRAME_DURATION = 75
FRAME_STRIDE = 2

FIG_WIDTH = 1200
FIG_HEIGHT = 800

# Optional: point this at a CSV with Date and Close columns.
# When None, the script uses Plotly's bundled AAPL historical index series.
CSV_PATH: Path | None = None
DATE_COLUMN = "Date"
PRICE_COLUMN = "Close"
TICKER_LABEL = "AAPL"

WRITE_HTML = True
SHOW_FIGURE = False


# ============================================================
# Styling — adapted from the supplied dark Plotly animation
# ============================================================

off_white = "#e0e0e0"
muted_white = "#aaaaaa"
axis_grid = "rgba(255,255,255,0.10)"
baseline_color = "#777777"

historical_red = "#ff4d5a"
historical_red_muted = "#ff9aa2"
backtest_green = "#29d17d"
backtest_green_muted = "#9af0c4"

axis_style = dict(
    showgrid=True,
    gridcolor=axis_grid,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# ============================================================
# Helpers
# ============================================================


def padded_range(values, pad_fraction=0.10, min_pad=3.0, floor=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)

    lower = low - pad
    upper = high + pad
    if floor is not None:
        lower = max(float(floor), lower)
    return [lower, upper]


def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


def load_historical_series() -> tuple[pd.DatetimeIndex, np.ndarray, str]:
    """Load a price series and normalize its first observation to 100."""
    if CSV_PATH is not None:
        frame = pd.read_csv(CSV_PATH)
        missing = {DATE_COLUMN, PRICE_COLUMN}.difference(frame.columns)
        if missing:
            raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

        frame = frame[[DATE_COLUMN, PRICE_COLUMN]].dropna().copy()
        frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN])
        frame = frame.sort_values(DATE_COLUMN)
        dates = pd.DatetimeIndex(frame[DATE_COLUMN])
        raw_prices = frame[PRICE_COLUMN].to_numpy(dtype=float)
        source_note = f"{TICKER_LABEL} from {CSV_PATH.name}"
    else:
        frame = px.data.stocks()[["date", "AAPL"]].dropna().copy()
        dates = pd.DatetimeIndex(pd.to_datetime(frame["date"]))
        raw_prices = frame["AAPL"].to_numpy(dtype=float)
        source_note = "AAPL from Plotly's bundled historical stocks dataset"

    if len(raw_prices) < 8:
        raise ValueError("At least eight historical observations are required.")
    if np.any(raw_prices <= 0):
        raise ValueError("All prices must be positive to construct log returns.")

    indexed_prices = 100.0 * raw_prices / raw_prices[0]
    return dates, indexed_prices, source_note


@dataclass(frozen=True)
class BacktestPath:
    name: str
    values: np.ndarray
    opacity: float
    tracking_error: float


def make_backtest_paths(
    historical_index: np.ndarray,
    seed: int = RANDOM_SEED,
) -> tuple[list[BacktestPath], np.ndarray, np.ndarray, np.ndarray]:
    """
    Create a visually tight family of perturbed equity curves.

    Each path starts from the historical log-return sequence and adds a small,
    autocorrelated perturbation. This preserves the broad shape while making
    every run distinct. Opacity increases as tracking error falls, so the paths
    that most closely agree with the central result appear more confidently.
    """
    rng = np.random.default_rng(seed)
    historical_log_returns = np.diff(np.log(historical_index))
    return_scale = max(float(np.std(historical_log_returns)), 1e-6)

    raw_paths = []
    for _ in range(N_BACKTEST_PATHS):
        innovation = rng.normal(0.0, return_scale * 0.11, len(historical_log_returns))
        perturbation = np.zeros_like(innovation)
        for i in range(len(innovation)):
            previous = perturbation[i - 1] if i else 0.0
            perturbation[i] = 0.68 * previous + innovation[i]

        # A tiny path-level drift difference prevents perfectly parallel lines.
        drift_tilt = rng.normal(0.0, return_scale * 0.008)
        adjusted_returns = historical_log_returns + perturbation + drift_tilt
        path = 100.0 * np.exp(np.r_[0.0, np.cumsum(adjusted_returns)])
        raw_paths.append(path)

    matrix = np.vstack(raw_paths)
    median_path = np.median(matrix, axis=0)
    lower_band = np.quantile(matrix, 0.10, axis=0)
    upper_band = np.quantile(matrix, 0.90, axis=0)

    errors = np.sqrt(np.mean((matrix - median_path) ** 2, axis=1))
    error_low, error_high = float(errors.min()), float(errors.max())
    denominator = max(error_high - error_low, 1e-9)

    paths = []
    for index, (values, error) in enumerate(zip(matrix, errors, strict=True), start=1):
        closeness = 1.0 - (float(error) - error_low) / denominator
        opacity = 0.10 + 0.46 * closeness
        paths.append(
            BacktestPath(
                name=f"Backtest {index:02d}",
                values=values,
                opacity=opacity,
                tracking_error=float(error),
            )
        )

    # Draw faint paths first and strong paths last to reinforce the cluster.
    paths.sort(key=lambda path: path.opacity)
    return paths, median_path, lower_band, upper_band


# ============================================================
# Data
# ============================================================


dates, historical_index, source_note = load_historical_series()
backtest_paths, median_backtest, lower_band, upper_band = make_backtest_paths(
    historical_index
)

all_values = np.concatenate(
    [
        historical_index,
        *(path.values for path in backtest_paths),
        median_backtest,
        lower_band,
        upper_band,
        np.array([100.0]),
    ]
)
y_range = padded_range(all_values, pad_fraction=0.08, min_pad=4.0, floor=0.0)

historical_return = 100.0 * (historical_index[-1] / historical_index[0] - 1.0)
median_return = 100.0 * (median_backtest[-1] / median_backtest[0] - 1.0)
median_band_width = float(np.median(upper_band - lower_band))
date_padding = pd.Timedelta(days=28)


# ============================================================
# Figure layout: one row x two columns
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}]],
    column_widths=[0.46, 0.54],
    horizontal_spacing=0.09,
    subplot_titles=(
        f"{TICKER_LABEL} historical price index<br><sup>Observed path, first value = 100</sup>",
        "Perturbed backtest equity curves<br><sup>Tighter agreement is emphasized with higher opacity</sup>",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================


def historical_trace(n_points: int):
    visible = historical_index[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=historical_red, width=4),
        text=endpoint_text(f"  History {visible[-1]:.1f}", n_points),
        textposition="middle right",
        textfont=dict(color=historical_red, size=11),
        name="Historical price index",
        legendgroup="historical",
        showlegend=True,
        customdata=100.0 * (visible / visible[0] - 1.0),
        hovertemplate=(
            f"<b>{TICKER_LABEL} history</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Price index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )


def backtest_trace(path: BacktestPath, n_points: int, showlegend: bool):
    visible = path.values[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines",
        line=dict(color=backtest_green, width=1.7),
        opacity=path.opacity,
        name="Perturbed backtest paths",
        legendgroup="backtests",
        showlegend=showlegend,
        customdata=100.0 * (visible / visible[0] - 1.0),
        hovertemplate=(
            f"<b>{path.name}</b><br>"
            f"Tracking error: {path.tracking_error:.2f} index points<br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Equity index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )


def median_backtest_trace(n_points: int):
    visible = median_backtest[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=backtest_green, width=4),
        text=endpoint_text(f"  Median {visible[-1]:.1f}", n_points),
        textposition="middle right",
        textfont=dict(color=backtest_green, size=11),
        name="Median backtest",
        legendgroup="backtest_median",
        showlegend=True,
        customdata=100.0 * (visible / visible[0] - 1.0),
        hovertemplate=(
            "<b>Median backtest</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Equity index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )


def traces_for_n(n_points: int):
    traces = [historical_trace(n_points)]
    for index, path in enumerate(backtest_paths):
        traces.append(backtest_trace(path, n_points, showlegend=(index == 0)))
    traces.append(median_backtest_trace(n_points))
    return traces


# ============================================================
# Initial traces, in the exact order reused by every frame
# ============================================================


fig.add_trace(historical_trace(initial_n), row=1, col=1)
for index, path in enumerate(backtest_paths):
    fig.add_trace(
        backtest_trace(path, initial_n, showlegend=(index == 0)),
        row=1,
        col=2,
    )
fig.add_trace(median_backtest_trace(initial_n), row=1, col=2)

# Starting-value reference lines.
for col in (1, 2):
    fig.add_hline(
        y=100.0,
        line=dict(color=baseline_color, width=1.4, dash="dot"),
        opacity=0.85,
        row=1,
        col=col,
    )


# ============================================================
# Animation frames
# ============================================================


frames = []
slider_steps = []
frame_positions = sorted(set(range(0, len(dates), FRAME_STRIDE)) | {len(dates) - 1})

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_data = traces_for_n(n_points)
    frame_name = f"observation_{frame_position}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %Y"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


for col in (1, 2):
    fig.update_xaxes(
        axis_style,
        row=1,
        col=col,
        range=[dates[0], dates[-1] + date_padding],
        title_text="Historical date",
        title_standoff=20,
        tickformat="%Y",
        dtick="M12",
    )
    fig.update_yaxes(
        axis_style,
        row=1,
        col=col,
        range=y_range,
        title_text="Indexed value (start = 100)",
        ticksuffix="",
    )

fig.update_layout(
    title=dict(
        text=(
            "Historical Price vs. Backtest Stability"
            "<br><sup>Observed history on the left | 24 lightly perturbed backtest paths on the right "
            "| brighter paths sit closer to the consensus</sup>"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=135, b=205, r=60, l=75),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.16,
        yanchor="top",
        font=dict(color=off_white),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.30,
            "yanchor": "top",
            "pad": {"r": 10, "t": 36},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.30,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Through: ",
                "font": {"color": muted_white},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=14))

fig.add_annotation(
    xref="x2 domain",
    yref="y2 domain",
    x=0.03,
    y=0.97,
    xanchor="left",
    yanchor="top",
    align="left",
    showarrow=False,
    text=(
        "<b>Robustness cue</b><br>"
        f"Median backtest return: {median_return:+.1f}%<br>"
        f"Historical return: {historical_return:+.1f}%<br>"
        f"Median 10–90% band width: {median_band_width:.1f} index points"
    ),
    font=dict(color=off_white, size=11),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.16)",
    borderwidth=1,
    borderpad=6,
)

# Note: Removed small text about perturbation family near slider as instructed.

###### ______________________________________________________________________________________________________________________________________

##### ᯓ⚾️ Maybe it does, maybe it does not

How should we think about this problem?  

Well what's the difference between picking a baseball team you hear is popular to win and running stats to pick a favorite?

Fundamentally, that's the difference between technical analysis and algorithmic trading.

The efficacy of each approach depends entirely on your sample and how *good* you are at each, this of course dictates your edge.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 23
N_PATHS = 24
N_PERIODS = 756  # roughly 3 years of trading days
FRAME_DURATION = 70
FRAME_STRIDE = 6

STARTING_WEALTH = 100.0
FIG_WIDTH = 1200
FIG_HEIGHT = 800

OUTPUT_HTML = Path("/mnt/data/trader_positive_ev_animation_v2.html")
WRITE_HTML = True
SHOW_FIGURE = False


# ============================================================
# Styling — adapted from the supplied dark Plotly animation
# ============================================================

off_white = "#e0e0e0"
muted_white = "#aaaaaa"
axis_grid = "rgba(255,255,255,0.10)"
baseline_color = "#777777"

path_green = "#29d17d"
path_green_muted = "#9af0c4"
path_red = "#ff4d5a"
path_red_muted = "#ff9aa2"

axis_style = dict(
    showgrid=True,
    gridcolor=axis_grid,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# ============================================================
# Helpers
# ============================================================


def padded_range(values, pad_fraction=0.10, min_pad=4.0, floor=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)

    lower = low - pad
    upper = high + pad
    if floor is not None:
        lower = max(float(floor), lower)
    return [lower, upper]



def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]



def split_around_threshold(values: np.ndarray, threshold: float) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    above = np.where(values >= threshold, values, np.nan)
    below = np.where(values < threshold, values, np.nan)
    return above, below


@dataclass(frozen=True)
class TraderProfile:
    name: str
    edge_per_period: float
    volatility: float
    persistence: float
    subtitle: str


@dataclass(frozen=True)
class WealthPath:
    name: str
    values: np.ndarray
    opacity: float
    tracking_error: float


TRADERS = [
    TraderProfile(
        name="Technical trader",
        edge_per_period=0.00080,
        volatility=0.0175,
        persistence=0.22,
        subtitle="Positive edge, but a looser realized path",
    ),
    TraderProfile(
        name="Algorithmic trader",
        edge_per_period=0.00070,
        volatility=0.0115,
        persistence=0.10,
        subtitle="Positive edge, with tighter implementation noise",
    ),
]


# ============================================================
# Simulate positive-EV wealth paths
# ============================================================


def business_dates() -> pd.DatetimeIndex:
    return pd.bdate_range(start="2020-01-02", periods=N_PERIODS)



def simulate_paths(
    profile: TraderProfile,
    seed_offset: int,
) -> tuple[list[WealthPath], np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Simulate a family of wealth paths with positive expected value.

    Each path compounds a small positive edge, but realized returns remain noisy.
    Noise is mildly autocorrelated to make the paths look organic rather than purely
    white-noise jagged. Opacity is stronger for paths that sit closer to the median.
    """
    rng = np.random.default_rng(RANDOM_SEED + seed_offset)
    n_returns = N_PERIODS - 1

    expected_path = STARTING_WEALTH * np.exp(
        np.r_[0.0, np.arange(1, N_PERIODS) * profile.edge_per_period]
    )

    raw_paths = []
    for _ in range(N_PATHS):
        innovation = rng.normal(0.0, profile.volatility, n_returns)
        perturbation = np.zeros(n_returns, dtype=float)

        for i in range(n_returns):
            previous = perturbation[i - 1] if i else 0.0
            perturbation[i] = profile.persistence * previous + innovation[i]

        drift_tilt = rng.normal(0.0, profile.volatility * 0.045)
        log_returns = profile.edge_per_period + perturbation + drift_tilt
        wealth = STARTING_WEALTH * np.exp(np.r_[0.0, np.cumsum(log_returns)])
        raw_paths.append(wealth)

    matrix = np.vstack(raw_paths)
    median_path = np.median(matrix, axis=0)
    lower_band = np.quantile(matrix, 0.10, axis=0)
    upper_band = np.quantile(matrix, 0.90, axis=0)

    errors = np.sqrt(np.mean((matrix - median_path) ** 2, axis=1))
    error_low, error_high = float(errors.min()), float(errors.max())
    denominator = max(error_high - error_low, 1e-9)

    paths = []
    for idx, (values, error) in enumerate(zip(matrix, errors, strict=True), start=1):
        closeness = 1.0 - (float(error) - error_low) / denominator
        opacity = 0.10 + 0.48 * closeness
        paths.append(
            WealthPath(
                name=f"{profile.name} path {idx:02d}",
                values=values,
                opacity=opacity,
                tracking_error=float(error),
            )
        )

    paths.sort(key=lambda path: path.opacity)
    return paths, median_path, expected_path, lower_band, upper_band


# ============================================================
# Data
# ============================================================


dates = business_dates()

technical_paths, technical_median, technical_expected, technical_lower, technical_upper = simulate_paths(
    TRADERS[0],
    seed_offset=0,
)
algorithmic_paths, algorithmic_median, algorithmic_expected, algorithmic_lower, algorithmic_upper = simulate_paths(
    TRADERS[1],
    seed_offset=10_000,
)

all_values = np.concatenate(
    [
        *(path.values for path in technical_paths),
        *(path.values for path in algorithmic_paths),
        technical_median,
        algorithmic_median,
        technical_expected,
        algorithmic_expected,
        technical_lower,
        technical_upper,
        algorithmic_lower,
        algorithmic_upper,
        np.array([STARTING_WEALTH]),
    ]
)
y_range = padded_range(all_values, pad_fraction=0.08, min_pad=5.0, floor=0.0)
date_padding = pd.Timedelta(days=28)

technical_median_return = 100.0 * (technical_median[-1] / technical_median[0] - 1.0)
technical_expected_return = 100.0 * (technical_expected[-1] / technical_expected[0] - 1.0)
algorithmic_median_return = 100.0 * (algorithmic_median[-1] / algorithmic_median[0] - 1.0)
algorithmic_expected_return = 100.0 * (algorithmic_expected[-1] / algorithmic_expected[0] - 1.0)

technical_band_width = float(np.median(technical_upper - technical_lower))
algorithmic_band_width = float(np.median(algorithmic_upper - algorithmic_lower))


# ============================================================
# Figure layout: one row x two columns
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}]],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.09,
    subplot_titles=(
        "Technical trader<br><sup>Positive EV can compound, but the realized path remains noisy</sup>",
        "Algorithmic trader<br><sup>Positive EV can compound, but the realized path still is not predetermined</sup>",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================


def path_traces(path: WealthPath, n_points: int, showlegend_green: bool, showlegend_red: bool):
    visible = path.values[:n_points]
    above, below = split_around_threshold(visible, STARTING_WEALTH)
    customdata = 100.0 * (visible / visible[0] - 1.0)

    green_trace = go.Scatter(
        x=dates[:n_points],
        y=above,
        mode="lines",
        line=dict(color=path_green, width=1.7),
        opacity=path.opacity,
        name="Positive EV paths",
        legendgroup="paths_green",
        showlegend=showlegend_green,
        customdata=customdata,
        hovertemplate=(
            f"<b>{path.name}</b><br>"
            f"Tracking error: {path.tracking_error:.2f} wealth points<br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    red_trace = go.Scatter(
        x=dates[:n_points],
        y=below,
        mode="lines",
        line=dict(color=path_red, width=1.7),
        opacity=path.opacity,
        name="Below starting principal",
        legendgroup="paths_red",
        showlegend=showlegend_red,
        customdata=customdata,
        hovertemplate=(
            f"<b>{path.name}</b><br>"
            f"Tracking error: {path.tracking_error:.2f} wealth points<br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    return [green_trace, red_trace]



def median_traces(values: np.ndarray, n_points: int, showlegend: bool, label: str):
    visible = values[:n_points]
    above, below = split_around_threshold(visible, STARTING_WEALTH)
    customdata = 100.0 * (visible / visible[0] - 1.0)
    final_above = visible[-1] >= STARTING_WEALTH

    green_trace = go.Scatter(
        x=dates[:n_points],
        y=above,
        mode="lines",
        line=dict(color=path_green, width=4),
        name="Median realized path",
        legendgroup="median",
        showlegend=showlegend,
        customdata=customdata,
        hovertemplate=(
            "<b>Median realized path</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    red_trace = go.Scatter(
        x=dates[:n_points],
        y=below,
        mode="lines",
        line=dict(color=path_red, width=4),
        name="Median realized path (sub-principal)",
        legendgroup="median_sub",
        showlegend=False,
        customdata=customdata,
        hovertemplate=(
            "<b>Median realized path</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth index: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    return [green_trace, red_trace]



def expected_traces(values: np.ndarray, n_points: int, showlegend: bool):
    visible = values[:n_points]
    above, below = split_around_threshold(visible, STARTING_WEALTH)
    customdata = 100.0 * (visible / visible[0] - 1.0)

    green_trace = go.Scatter(
        x=dates[:n_points],
        y=above,
        mode="lines",
        line=dict(color=path_green_muted, width=3, dash="dash"),
        name="Expected-value drift",
        legendgroup="expected",
        showlegend=showlegend,
        customdata=customdata,
        hovertemplate=(
            "<b>Expected-value drift</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Expected wealth index: %{y:.2f}<br>"
            "Expected return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    red_trace = go.Scatter(
        x=dates[:n_points],
        y=below,
        mode="lines",
        line=dict(color=path_red_muted, width=3, dash="dash"),
        name="Expected-value drift (sub-principal)",
        legendgroup="expected_sub",
        showlegend=False,
        customdata=customdata,
        hovertemplate=(
            "<b>Expected-value drift</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Expected wealth index: %{y:.2f}<br>"
            "Expected return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )

    return [green_trace, red_trace]



def traces_for_n(n_points: int):
    traces = []

    for idx, path in enumerate(technical_paths):
        traces.extend(path_traces(path, n_points, showlegend_green=(idx == 0), showlegend_red=(idx == 0)))
    traces.extend(median_traces(technical_median, n_points, showlegend=True, label="Median"))
    traces.extend(expected_traces(technical_expected, n_points, showlegend=True))

    for idx, path in enumerate(algorithmic_paths):
        traces.extend(path_traces(path, n_points, showlegend_green=False, showlegend_red=False))
    traces.extend(median_traces(algorithmic_median, n_points, showlegend=False, label="Median"))
    traces.extend(expected_traces(algorithmic_expected, n_points, showlegend=False))

    return traces


# ============================================================
# Initial traces, in the exact order reused by every frame
# ============================================================


for idx, path in enumerate(technical_paths):
    for trace in path_traces(path, initial_n, showlegend_green=(idx == 0), showlegend_red=(idx == 0)):
        fig.add_trace(trace, row=1, col=1)
for trace in median_traces(technical_median, initial_n, showlegend=True, label="Median"):
    fig.add_trace(trace, row=1, col=1)
for trace in expected_traces(technical_expected, initial_n, showlegend=True):
    fig.add_trace(trace, row=1, col=1)

for idx, path in enumerate(algorithmic_paths):
    for trace in path_traces(path, initial_n, showlegend_green=False, showlegend_red=False):
        fig.add_trace(trace, row=1, col=2)
for trace in median_traces(algorithmic_median, initial_n, showlegend=False, label="Median"):
    fig.add_trace(trace, row=1, col=2)
for trace in expected_traces(algorithmic_expected, initial_n, showlegend=False):
    fig.add_trace(trace, row=1, col=2)

for col in (1, 2):
    fig.add_hline(
        y=STARTING_WEALTH,
        line=dict(color=baseline_color, width=1.4, dash="dot"),
        opacity=0.85,
        row=1,
        col=col,
    )


# ============================================================
# Animation frames
# ============================================================


frames = []
slider_steps = []
frame_positions = sorted(set(range(0, len(dates), FRAME_STRIDE)) | {len(dates) - 1})

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_data = traces_for_n(n_points)
    frame_name = f"observation_{frame_position}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %Y"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


for col in (1, 2):
    fig.update_xaxes(
        axis_style,
        row=1,
        col=col,
        range=[dates[0], dates[-1] + date_padding],
        title_text="Date",
        title_standoff=20,
        tickformat="%Y",
        dtick="M12",
    )
    fig.update_yaxes(
        axis_style,
        row=1,
        col=col,
        range=y_range,
        title_text="Wealth index (start = 100)",
    )

fig.update_layout(
    title=dict(
        text=(
            "Positive EV Is Not Path Control"
            "<br><sup>Technical and algorithmic traders may have an edge, "
            "but realized wealth remains stochastic — edge shapes the distribution, not the exact walk</sup>"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=138, b=210, r=60, l=75),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.16,
        yanchor="top",
        font=dict(color=off_white),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.30,
            "yanchor": "top",
            "pad": {"r": 10, "t": 36},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.30,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Through: ",
                "font": {"color": muted_white},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=14))

# -------- Removed plot annotations / text overlays as instructed --------

###### ______________________________________________________________________________________________________________________________________

##### 🎲 Most of the time it's your fault, but sometimes it's not

Some things are in your control as a technical or algorithmic trader, other things aren't.

Your action is in your control, but the way the market reacts is not.  Both are distinct sources of variance that impact your trajectory.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 31
N_TRADING_DAYS = 252  # one trading year
N_SIMULATIONS = 500
FRAME_DURATION = 70
FRAME_STRIDE = 3

STARTING_WEALTH = 100.0
FIG_WIDTH = 1200
FIG_HEIGHT = 800

OUTPUT_HTML = Path("/mnt/data/trader_variance_cone_v2.html")
WRITE_HTML = True
SHOW_FIGURE = False


# ============================================================
# Styling — consistent with the prior dark Plotly look
# ============================================================

off_white = "#e0e0e0"
muted_white = "#aaaaaa"
axis_grid = "rgba(255,255,255,0.10)"
baseline_color = "#777777"

technical_green = "#29d17d"
technical_green_muted = "rgba(41, 209, 125, 0.24)"
technical_green_inner = "rgba(41, 209, 125, 0.38)"
algorithmic_cyan = "#4dd6ff"
algorithmic_cyan_muted = "rgba(77, 214, 255, 0.22)"
algorithmic_cyan_inner = "rgba(77, 214, 255, 0.36)"
expected_muted = "#c9d1d9"
stress_gray = "rgba(255,255,255,0.06)"

axis_style = dict(
    showgrid=True,
    gridcolor=axis_grid,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# ============================================================
# Helpers
# ============================================================


def padded_range(values, pad_fraction=0.10, min_pad=4.0, floor=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)

    lower = low - pad
    upper = high + pad
    if floor is not None:
        lower = max(float(floor), lower)
    return [lower, upper]



def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


@dataclass(frozen=True)
class TraderProfile:
    name: str
    edge_per_day: float
    market_exposure: float
    trader_noise: float
    adaptability: float
    persistence: float
    cone_color: str
    outer_fill: str
    inner_fill: str
    note: str


@dataclass(frozen=True)
class TraderSimulation:
    paths: np.ndarray
    realized_path: np.ndarray
    expected_path: np.ndarray
    inner_lower: np.ndarray
    inner_upper: np.ndarray
    outer_lower: np.ndarray
    outer_upper: np.ndarray
    market_variance_share: float
    trader_variance_share: float
    realized_return: float
    expected_return: float
    median_cone_width: float


TECHNICAL = TraderProfile(
    name="Technical trader",
    edge_per_day=0.00055,
    market_exposure=1.00,
    trader_noise=0.0095,
    adaptability=0.18,
    persistence=0.28,
    cone_color=technical_green,
    outer_fill=technical_green_muted,
    inner_fill=technical_green_inner,
    note="Broader cone: greater sensitivity to market swings and more trader-specific noise",
)

ALGORITHMIC = TraderProfile(
    name="Algorithmic trader",
    edge_per_day=0.00058,
    market_exposure=0.86,
    trader_noise=0.0048,
    adaptability=0.58,
    persistence=0.12,
    cone_color=algorithmic_cyan,
    outer_fill=algorithmic_cyan_muted,
    inner_fill=algorithmic_cyan_inner,
    note="Tighter cone: adaptive sizing dampens market shocks and reduces implementation noise",
)


# ============================================================
# Market dynamics
# ============================================================


def business_dates() -> pd.DatetimeIndex:
    return pd.bdate_range(start="2024-01-02", periods=N_TRADING_DAYS)



def market_regime_profile() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a stylized one-year market environment.

    The year contains a calm start, a pronounced mid-year volatility spike,
    and a later recovery regime. The volatility state widens the cone for both
    traders, while adaptability determines how much each trader dampens that shock.
    """
    t = np.arange(N_TRADING_DAYS - 1, dtype=float)

    # Smooth volatility state: quiet -> stressed -> easing.
    stress_peak = np.exp(-0.5 * ((t - 112.0) / 24.0) ** 2)
    aftershock = 0.55 * np.exp(-0.5 * ((t - 168.0) / 30.0) ** 2)
    vol_state = np.clip(stress_peak + aftershock, 0.0, None)
    vol_state /= vol_state.max()

    market_sigma = 0.0058 + 0.0105 * vol_state

    # Mild trend structure: early drift, drawdown pocket, then recovery.
    market_mu = 0.00010 - 0.00095 * stress_peak + 0.00035 * np.exp(-0.5 * ((t - 205.0) / 34.0) ** 2)
    return t, market_mu, market_sigma



def simulate_trader(profile: TraderProfile, seed_offset: int) -> TraderSimulation:
    rng = np.random.default_rng(RANDOM_SEED + seed_offset)
    _, market_mu, market_sigma = market_regime_profile()
    n_returns = N_TRADING_DAYS - 1

    vol_state = (market_sigma - market_sigma.min()) / max(market_sigma.max() - market_sigma.min(), 1e-9)
    effective_market_sigma = profile.market_exposure * market_sigma * (1.0 - profile.adaptability * vol_state)

    def one_path() -> np.ndarray:
        market_shock = rng.normal(market_mu, market_sigma)
        effective_market = effective_market_sigma * ((market_shock - market_mu) / np.maximum(market_sigma, 1e-12))

        trader_innovation = rng.normal(0.0, profile.trader_noise, n_returns)
        trader_component = np.zeros(n_returns, dtype=float)
        for i in range(n_returns):
            prev = trader_component[i - 1] if i else 0.0
            trader_component[i] = profile.persistence * prev + trader_innovation[i]

        log_returns = profile.edge_per_day + market_mu + effective_market + trader_component
        return STARTING_WEALTH * np.exp(np.r_[0.0, np.cumsum(log_returns)])

    paths = np.vstack([one_path() for _ in range(N_SIMULATIONS)])
    expected_path = paths.mean(axis=0)
    inner_lower = np.quantile(paths, 0.25, axis=0)
    inner_upper = np.quantile(paths, 0.75, axis=0)
    outer_lower = np.quantile(paths, 0.10, axis=0)
    outer_upper = np.quantile(paths, 0.90, axis=0)

    # Choose one illustrative path that experiences the shared stress regime clearly,
    # but remains representative rather than extreme.
    path_scores = np.mean(np.abs(paths - expected_path), axis=1)
    ranked = np.argsort(path_scores)
    realized_path = paths[ranked[len(ranked) // 3]]

    market_var = np.mean(effective_market_sigma**2)
    trader_var = profile.trader_noise**2 / max(1.0 - profile.persistence**2, 1e-9)
    total_var = market_var + trader_var
    market_share = 100.0 * market_var / total_var
    trader_share = 100.0 * trader_var / total_var

    realized_return = 100.0 * (realized_path[-1] / realized_path[0] - 1.0)
    expected_return = 100.0 * (expected_path[-1] / expected_path[0] - 1.0)
    median_cone_width = float(np.median(outer_upper - outer_lower))

    return TraderSimulation(
        paths=paths,
        realized_path=realized_path,
        expected_path=expected_path,
        inner_lower=inner_lower,
        inner_upper=inner_upper,
        outer_lower=outer_lower,
        outer_upper=outer_upper,
        market_variance_share=market_share,
        trader_variance_share=trader_share,
        realized_return=realized_return,
        expected_return=expected_return,
        median_cone_width=median_cone_width,
    )


# ============================================================
# Data
# ============================================================


dates = business_dates()
_, market_mu, market_sigma = market_regime_profile()
tech_sim = simulate_trader(TECHNICAL, seed_offset=0)
algo_sim = simulate_trader(ALGORITHMIC, seed_offset=10_000)

all_values = np.concatenate(
    [
        tech_sim.realized_path,
        tech_sim.expected_path,
        tech_sim.outer_lower,
        tech_sim.outer_upper,
        algo_sim.realized_path,
        algo_sim.expected_path,
        algo_sim.outer_lower,
        algo_sim.outer_upper,
        np.array([STARTING_WEALTH]),
    ]
)
y_range = padded_range(all_values, pad_fraction=0.08, min_pad=5.0, floor=0.0)

date_padding = pd.Timedelta(days=10)
stress_idx = int(np.argmax(market_sigma))
stress_start = max(0, stress_idx - 24)
stress_end = min(len(dates) - 1, stress_idx + 24)


# ============================================================
# Figure layout
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}]],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.09,
    subplot_titles=(
        "Technical trader<br><sup>Wider variance cone under shifting market regimes</sup>",
        "Algorithmic trader<br><sup>Adaptive response narrows the variance cone</sup>",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================


def cone_traces(sim: TraderSimulation, profile: TraderProfile, n_points: int, showlegend_outer: bool, showlegend_inner: bool):
    x = dates[:n_points]
    traces = [
        go.Scatter(
            x=x,
            y=sim.outer_upper[:n_points],
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name="",
        ),
        go.Scatter(
            x=x,
            y=sim.outer_lower[:n_points],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=profile.outer_fill,
            name="80% variance cone",
            legendgroup=f"outer_{profile.name}",
            showlegend=showlegend_outer,
            hovertemplate=(
                "<b>80% cone</b><br>"
                "Date: %{x|%b %d, %Y}<br>"
                "Lower bound: %{y:.2f}<extra></extra>"
            ),
        ),
        go.Scatter(
            x=x,
            y=sim.inner_upper[:n_points],
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name="",
        ),
        go.Scatter(
            x=x,
            y=sim.inner_lower[:n_points],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=profile.inner_fill,
            name="50% variance cone",
            legendgroup=f"inner_{profile.name}",
            showlegend=showlegend_inner,
            hovertemplate=(
                "<b>50% cone</b><br>"
                "Date: %{x|%b %d, %Y}<br>"
                "Lower bound: %{y:.2f}<extra></extra>"
            ),
        ),
    ]
    return traces



def expected_trace(sim: TraderSimulation, profile: TraderProfile, n_points: int, showlegend: bool):
    visible = sim.expected_path[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines",  # removed '+text'
        line=dict(color=expected_muted, width=3, dash="dash"),
        # removed text and textposition
        # removed textfont
        name="Expected path",
        legendgroup=f"expected_{profile.name}",
        showlegend=showlegend,
        customdata=100.0 * (visible / visible[0] - 1.0),
        hovertemplate=(
            "<b>Expected path</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Expected wealth: %{y:.2f}<br>"
            "Expected return: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )



def realized_trace(sim: TraderSimulation, profile: TraderProfile, n_points: int, showlegend: bool):
    visible = sim.realized_path[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines",  # removed '+text'
        line=dict(color=profile.cone_color, width=4),
        # removed text and textposition
        # removed textfont
        name="Illustrative realized path",
        legendgroup=f"realized_{profile.name}",
        showlegend=showlegend,
        customdata=100.0 * (visible / visible[0] - 1.0),
        hovertemplate=(
            f"<b>{profile.name}</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth: %{y:.2f}<br>"
            "Return from start: %{customdata:+.1f}%"
            "<extra></extra>"
        ),
    )



def traces_for_n(n_points: int):
    traces = []
    traces.extend(cone_traces(tech_sim, TECHNICAL, n_points, showlegend_outer=True, showlegend_inner=True))
    traces.append(expected_trace(tech_sim, TECHNICAL, n_points, showlegend=True))
    traces.append(realized_trace(tech_sim, TECHNICAL, n_points, showlegend=True))

    traces.extend(cone_traces(algo_sim, ALGORITHMIC, n_points, showlegend_outer=False, showlegend_inner=False))
    traces.append(expected_trace(algo_sim, ALGORITHMIC, n_points, showlegend=False))
    traces.append(realized_trace(algo_sim, ALGORITHMIC, n_points, showlegend=False))
    return traces


# ============================================================
# Initial traces
# ============================================================


for trace in cone_traces(tech_sim, TECHNICAL, initial_n, showlegend_outer=True, showlegend_inner=True):
    fig.add_trace(trace, row=1, col=1)
fig.add_trace(expected_trace(tech_sim, TECHNICAL, initial_n, showlegend=True), row=1, col=1)
fig.add_trace(realized_trace(tech_sim, TECHNICAL, initial_n, showlegend=True), row=1, col=1)

for trace in cone_traces(algo_sim, ALGORITHMIC, initial_n, showlegend_outer=False, showlegend_inner=False):
    fig.add_trace(trace, row=1, col=2)
fig.add_trace(expected_trace(algo_sim, ALGORITHMIC, initial_n, showlegend=False), row=1, col=2)
fig.add_trace(realized_trace(algo_sim, ALGORITHMIC, initial_n, showlegend=False), row=1, col=2)

for col in (1, 2):
    fig.add_hline(
        y=STARTING_WEALTH,
        line=dict(color=baseline_color, width=1.4, dash="dot"),
        opacity=0.85,
        row=1,
        col=col,
    )
    fig.add_vrect(
        x0=dates[stress_start],
        x1=dates[stress_end],
        fillcolor=stress_gray,
        opacity=1.0,
        line_width=0,
        row=1,
        col=col,
    )


# ============================================================
# Animation frames
# ============================================================


frames = []
slider_steps = []
frame_positions = sorted(set(range(0, len(dates), FRAME_STRIDE)) | {len(dates) - 1})

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_data = traces_for_n(n_points)
    frame_name = f"observation_{frame_position}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %d"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


for col in (1, 2):
    fig.update_xaxes(
        axis_style,
        row=1,
        col=col,
        range=[dates[0], dates[-1] + date_padding],
        title_text="Date",
        title_standoff=20,
        tickformat="%b",
        dtick="M1",
    )
    fig.update_yaxes(
        axis_style,
        row=1,
        col=col,
        range=y_range,
        title_text="Wealth index (start = 100)",
    )

fig.update_layout(
    title=dict(
        text=(
            "One-Year Variance Cone: Expectation vs. Realized Path"
            "<br><sup>Deviations from expectation are driven by both the trader and the market — "
            "the market stress regime widens the cone, while trader design shapes how tightly outcomes are controlled</sup>"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=145, b=220, r=60, l=75),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
        yanchor="top",
        font=dict(color=off_white),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.33,
            "yanchor": "top",
            "pad": {"r": 10, "t": 36},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.33,
            "yanchor": "top",
            "currentvalue": {"prefix": "Through: ", "font": {"color": muted_white}},
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=14))

# The following chart overlays have been removed per instruction:
# - Technical trader overlay
# - Algorithmic trader overlay
# - Gray band explanation overlay

----

##### 🎰 Is there any reason to believe that either will work in the future?

The unsexy truth is that anything *can* work, it is whether or not you will make it to the long run applying it as a strategy.

If you happen to be *really* good at calling "support" and "resistance" regions and dynamically updating your positions, you may have an edge.

If you happen to be *really* good at identifying statistical structure, and structural breaks dynamically updating your positions, you may have an edge.

The key thing to understand is that it is about *performance*.  

You can be the best baseball player in the world and still have *bad* performance.  The difference between a good player and a bad player is if they have an edge.  Their edge could be working harder, continuing to learn, the list goes on...but it's never stagnation.  

You can't get good and stay good but just sitting still.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 47
N_TRADING_DAYS = 252  # one trading year
N_SIMULATIONS = 500
FRAME_DURATION = 70
FRAME_STRIDE = 3

STARTING_WEALTH = 100.0
REGIME_BREAK_IDX = 112
ADAPT_END_IDX = 146

FIG_WIDTH = 1200
FIG_HEIGHT = 800

OUTPUT_HTML = Path("/mnt/data/static_vs_dynamic_trader_v3.html")
WRITE_HTML = True
SHOW_FIGURE = False


# ============================================================
# Styling — matched to the supplied variance-cone animation
# ============================================================

off_white = "#e0e0e0"
muted_white = "#aaaaaa"
axis_grid = "rgba(255,255,255,0.10)"
baseline_color = "#777777"

static_orange = "#ff9f43"
static_outer_fill = "rgba(255, 159, 67, 0.22)"
static_inner_fill = "rgba(255, 159, 67, 0.36)"

dynamic_cyan = "#4dd6ff"
dynamic_outer_fill = "rgba(77, 214, 255, 0.22)"
dynamic_inner_fill = "rgba(77, 214, 255, 0.36)"

expected_muted = "#c9d1d9"
stress_gray = "rgba(255,255,255,0.06)"
negative_edge_red = "#ff4d5a"
positive_edge_green = "#29d17d"

axis_style = dict(
    showgrid=True,
    gridcolor=axis_grid,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# ============================================================
# Helpers
# ============================================================


def padded_range(values, pad_fraction=0.10, min_pad=4.0, floor=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)

    lower = low - pad
    upper = high + pad
    if floor is not None:
        lower = max(float(floor), lower)
    return [lower, upper]



def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


@dataclass(frozen=True)
class TraderProfile:
    name: str
    market_exposure: float
    trader_noise: float
    adaptability: float
    persistence: float
    cone_color: str
    outer_fill: str
    inner_fill: str
    note: str


@dataclass(frozen=True)
class TraderSimulation:
    paths: np.ndarray
    realized_path: np.ndarray
    expected_path: np.ndarray
    edge_schedule: np.ndarray
    inner_lower: np.ndarray
    inner_upper: np.ndarray
    outer_lower: np.ndarray
    outer_upper: np.ndarray
    market_variance_share: float
    trader_variance_share: float
    realized_return: float
    expected_return: float
    median_cone_width: float
    post_break_return: float


STATIC = TraderProfile(
    name="Static trader",
    market_exposure=1.00,
    trader_noise=0.0078,
    adaptability=0.00,
    persistence=0.24,
    cone_color=static_orange,
    outer_fill=static_outer_fill,
    inner_fill=static_inner_fill,
    note="The strategy remains unchanged after the regime break, so its former edge becomes persistently negative.",
)

DYNAMIC = TraderProfile(
    name="Dynamic trader",
    market_exposure=0.94,
    trader_noise=0.0052,
    adaptability=0.62,
    persistence=0.12,
    cone_color=dynamic_cyan,
    outer_fill=dynamic_outer_fill,
    inner_fill=dynamic_inner_fill,
    note="The strategy takes a hit, updates during the transition, and restores positive edge in the new regime.",
)


# ============================================================
# Market dynamics and edge schedules
# ============================================================


def business_dates() -> pd.DatetimeIndex:
    return pd.bdate_range(start="2024-01-02", periods=N_TRADING_DAYS)



def market_regime_profile() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Stylized one-year environment with a mid-year regime change / news break.

    Volatility rises abruptly around the break, expected market returns weaken,
    and then the environment partially normalizes. Both traders see the same
    market structure; their strategy updates determine what happens to edge.
    """
    t = np.arange(N_TRADING_DAYS - 1, dtype=float)

    stress_peak = np.exp(-0.5 * ((t - REGIME_BREAK_IDX) / 18.0) ** 2)
    aftershock = 0.48 * np.exp(-0.5 * ((t - 166.0) / 28.0) ** 2)
    vol_state = np.clip(stress_peak + aftershock, 0.0, None)
    vol_state /= vol_state.max()

    market_sigma = 0.0058 + 0.0115 * vol_state
    market_mu = (
        0.00010
        - 0.00125 * stress_peak
        + 0.00034 * np.exp(-0.5 * ((t - 212.0) / 32.0) ** 2)
    )
    return t, market_mu, market_sigma



def static_edge_schedule() -> np.ndarray:
    t = np.arange(N_TRADING_DAYS - 1)
    return np.where(t < REGIME_BREAK_IDX, 0.00072, -0.00105)



def dynamic_edge_schedule() -> np.ndarray:
    t = np.arange(N_TRADING_DAYS - 1)
    return np.where(
        t < REGIME_BREAK_IDX,
        0.00068,
        np.where(t < ADAPT_END_IDX, -0.00078, 0.00102),
    )


# ============================================================
# Trader simulation
# ============================================================


def simulate_trader(
    profile: TraderProfile,
    edge_schedule: np.ndarray,
    seed_offset: int,
) -> TraderSimulation:
    rng = np.random.default_rng(RANDOM_SEED + seed_offset)
    _, market_mu, market_sigma = market_regime_profile()
    n_returns = N_TRADING_DAYS - 1

    vol_state = (market_sigma - market_sigma.min()) / max(
        market_sigma.max() - market_sigma.min(),
        1e-9,
    )
    effective_market_sigma = (
        profile.market_exposure
        * market_sigma
        * (1.0 - profile.adaptability * vol_state)
    )

    def one_path() -> np.ndarray:
        market_z = rng.normal(0.0, 1.0, n_returns)
        effective_market = market_mu + effective_market_sigma * market_z

        trader_innovation = rng.normal(0.0, profile.trader_noise, n_returns)
        trader_component = np.zeros(n_returns, dtype=float)
        for i in range(n_returns):
            previous = trader_component[i - 1] if i else 0.0
            trader_component[i] = (
                profile.persistence * previous + trader_innovation[i]
            )

        log_returns = edge_schedule + effective_market + trader_component
        return STARTING_WEALTH * np.exp(np.r_[0.0, np.cumsum(log_returns)])

    paths = np.vstack([one_path() for _ in range(N_SIMULATIONS)])

    # Deterministic expected path makes the edge transition explicit.
    expected_log_returns = edge_schedule + market_mu
    expected_path = STARTING_WEALTH * np.exp(
        np.r_[0.0, np.cumsum(expected_log_returns)]
    )

    inner_lower = np.quantile(paths, 0.25, axis=0)
    inner_upper = np.quantile(paths, 0.75, axis=0)
    outer_lower = np.quantile(paths, 0.10, axis=0)
    outer_upper = np.quantile(paths, 0.90, axis=0)

    # Select a representative path with a visible break response.
    break_drawdown = paths[:, ADAPT_END_IDX] / np.maximum(
        paths[:, REGIME_BREAK_IDX],
        1e-12,
    ) - 1.0
    terminal_distance = np.abs(paths[:, -1] - expected_path[-1])
    score = np.abs(break_drawdown + 0.055) + 0.010 * terminal_distance
    realized_path = paths[np.argmin(score)]

    market_var = np.mean(effective_market_sigma**2)
    trader_var = profile.trader_noise**2 / max(
        1.0 - profile.persistence**2,
        1e-9,
    )
    total_var = market_var + trader_var
    market_share = 100.0 * market_var / total_var
    trader_share = 100.0 * trader_var / total_var

    realized_return = 100.0 * (
        realized_path[-1] / realized_path[0] - 1.0
    )
    expected_return = 100.0 * (
        expected_path[-1] / expected_path[0] - 1.0
    )
    post_break_return = 100.0 * (
        realized_path[-1] / realized_path[REGIME_BREAK_IDX] - 1.0
    )
    median_cone_width = float(np.median(outer_upper - outer_lower))

    return TraderSimulation(
        paths=paths,
        realized_path=realized_path,
        expected_path=expected_path,
        edge_schedule=np.r_[edge_schedule[0], edge_schedule],
        inner_lower=inner_lower,
        inner_upper=inner_upper,
        outer_lower=outer_lower,
        outer_upper=outer_upper,
        market_variance_share=market_share,
        trader_variance_share=trader_share,
        realized_return=realized_return,
        expected_return=expected_return,
        median_cone_width=median_cone_width,
        post_break_return=post_break_return,
    )


# ============================================================
# Data
# ============================================================


dates = business_dates()
_, market_mu, market_sigma = market_regime_profile()

static_sim = simulate_trader(
    STATIC,
    edge_schedule=static_edge_schedule(),
    seed_offset=0,
)
dynamic_sim = simulate_trader(
    DYNAMIC,
    edge_schedule=dynamic_edge_schedule(),
    seed_offset=10_000,
)

all_values = np.concatenate(
    [
        static_sim.realized_path,
        static_sim.expected_path,
        static_sim.outer_lower,
        static_sim.outer_upper,
        dynamic_sim.realized_path,
        dynamic_sim.expected_path,
        dynamic_sim.outer_lower,
        dynamic_sim.outer_upper,
        np.array([STARTING_WEALTH]),
    ]
)
y_range = padded_range(all_values, pad_fraction=0.08, min_pad=5.0, floor=0.0)

date_padding = pd.Timedelta(days=10)
stress_idx = int(np.argmax(market_sigma))
stress_start = max(0, stress_idx - 22)
stress_end = min(len(dates) - 1, stress_idx + 22)


# ============================================================
# Figure layout
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}]],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.09,
    subplot_titles=(
        "Static trader",
        "Dynamic trader",
    ),
)

initial_n = 1


# ============================================================
# Trace builders — same ordering reused by every animation frame
# ============================================================


def cone_traces(
    sim: TraderSimulation,
    profile: TraderProfile,
    n_points: int,
    showlegend_outer: bool,
    showlegend_inner: bool,
):
    x = dates[:n_points]
    return [
        go.Scatter(
            x=x,
            y=sim.outer_upper[:n_points],
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name="",
        ),
        go.Scatter(
            x=x,
            y=sim.outer_lower[:n_points],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=profile.outer_fill,
            name="80% variance cone",
            legendgroup=f"outer_{profile.name}",
            showlegend=showlegend_outer,
            hovertemplate=(
                "<b>80% cone</b><br>"
                "Date: %{x|%b %d, %Y}<br>"
                "Lower bound: %{y:.2f}<extra></extra>"
            ),
        ),
        go.Scatter(
            x=x,
            y=sim.inner_upper[:n_points],
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name="",
        ),
        go.Scatter(
            x=x,
            y=sim.inner_lower[:n_points],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=profile.inner_fill,
            name="50% variance cone",
            legendgroup=f"inner_{profile.name}",
            showlegend=showlegend_inner,
            hovertemplate=(
                "<b>50% cone</b><br>"
                "Date: %{x|%b %d, %Y}<br>"
                "Lower bound: %{y:.2f}<extra></extra>"
            ),
        ),
    ]



def expected_trace(
    sim: TraderSimulation,
    profile: TraderProfile,
    n_points: int,
    showlegend: bool,
):
    visible = sim.expected_path[:n_points]
    visible_edge = sim.edge_schedule[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=expected_muted, width=3, dash="dash"),
        text=endpoint_text(f"  EV {visible[-1]:.1f}", n_points),
        textposition="middle right",
        textfont=dict(color=expected_muted, size=11),
        name="Expected path",
        legendgroup=f"expected_{profile.name}",
        showlegend=showlegend,
        customdata=np.column_stack(
            [
                100.0 * (visible / visible[0] - 1.0),
                visible_edge,
            ]
        ),
        hovertemplate=(
            "<b>Expected path</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Expected wealth: %{y:.2f}<br>"
            "Expected return: %{customdata[0]:+.1f}%<br>"
            "Strategy edge: %{customdata[1]:+.4f} per day"
            "<extra></extra>"
        ),
    )



def realized_trace(
    sim: TraderSimulation,
    profile: TraderProfile,
    n_points: int,
    showlegend: bool,
):
    visible = sim.realized_path[:n_points]
    visible_edge = sim.edge_schedule[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=profile.cone_color, width=4),
        text=endpoint_text(f"  Realized {visible[-1]:.1f}", n_points),
        textposition="middle right",
        textfont=dict(color=profile.cone_color, size=11),
        name="Illustrative realized path",
        legendgroup=f"realized_{profile.name}",
        showlegend=showlegend,
        customdata=np.column_stack(
            [
                100.0 * (visible / visible[0] - 1.0),
                visible_edge,
            ]
        ),
        hovertemplate=(
            f"<b>{profile.name}</b><br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Wealth: %{y:.2f}<br>"
            "Return from start: %{customdata[0]:+.1f}%<br>"
            "Strategy edge: %{customdata[1]:+.4f} per day"
            "<extra></extra>"
        ),
    )



def traces_for_n(n_points: int):
    traces = []

    traces.extend(
        cone_traces(
            static_sim,
            STATIC,
            n_points,
            showlegend_outer=True,
            showlegend_inner=True,
        )
    )
    traces.append(
        expected_trace(
            static_sim,
            STATIC,
            n_points,
            showlegend=True,
        )
    )
    traces.append(
        realized_trace(
            static_sim,
            STATIC,
            n_points,
            showlegend=True,
        )
    )

    traces.extend(
        cone_traces(
            dynamic_sim,
            DYNAMIC,
            n_points,
            showlegend_outer=False,
            showlegend_inner=False,
        )
    )
    traces.append(
        expected_trace(
            dynamic_sim,
            DYNAMIC,
            n_points,
            showlegend=False,
        )
    )
    traces.append(
        realized_trace(
            dynamic_sim,
            DYNAMIC,
            n_points,
            showlegend=False,
        )
    )

    return traces


# ============================================================
# Initial traces
# ============================================================


for trace in cone_traces(
    static_sim,
    STATIC,
    initial_n,
    showlegend_outer=True,
    showlegend_inner=True,
):
    fig.add_trace(trace, row=1, col=1)
fig.add_trace(
    expected_trace(static_sim, STATIC, initial_n, showlegend=True),
    row=1,
    col=1,
)
fig.add_trace(
    realized_trace(static_sim, STATIC, initial_n, showlegend=True),
    row=1,
    col=1,
)

for trace in cone_traces(
    dynamic_sim,
    DYNAMIC,
    initial_n,
    showlegend_outer=False,
    showlegend_inner=False,
):
    fig.add_trace(trace, row=1, col=2)
fig.add_trace(
    expected_trace(dynamic_sim, DYNAMIC, initial_n, showlegend=False),
    row=1,
    col=2,
)
fig.add_trace(
    realized_trace(dynamic_sim, DYNAMIC, initial_n, showlegend=False),
    row=1,
    col=2,
)

for col in (1, 2):
    fig.add_hline(
        y=STARTING_WEALTH,
        line=dict(color=baseline_color, width=1.4, dash="dot"),
        opacity=0.85,
        row=1,
        col=col,
    )
    fig.add_vrect(
        x0=dates[stress_start],
        x1=dates[stress_end],
        fillcolor=stress_gray,
        opacity=1.0,
        line_width=0,
        row=1,
        col=col,
    )

# The dynamic trader's adaptation completion is explicitly marked.
fig.add_vline(
    x=dates[ADAPT_END_IDX],
    line=dict(color=dynamic_cyan, width=1.4, dash="dot"),
    opacity=0.70,
    row=1,
    col=2,
)


# ============================================================
# Animation frames
# ============================================================


frames = []
slider_steps = []
frame_positions = sorted(
    set(range(0, len(dates), FRAME_STRIDE))
    | {REGIME_BREAK_IDX, ADAPT_END_IDX, len(dates) - 1}
)

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_data = traces_for_n(n_points)
    frame_name = f"observation_{frame_position}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %d"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


for col in (1, 2):
    fig.update_xaxes(
        axis_style,
        row=1,
        col=col,
        range=[dates[0], dates[-1] + date_padding],
        title_text="Date",
        title_standoff=20,
        tickformat="%b",
        dtick="M1",
    )
    fig.update_yaxes(
        axis_style,
        row=1,
        col=col,
        range=y_range,
        title_text="Wealth index (start = 100)",
    )

fig.update_layout(
    title=dict(
        text=(
            "Static vs. Dynamic Trader: Edge Through a Regime Break"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=145, b=220, r=60, l=75),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
        yanchor="top",
        font=dict(color=off_white),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.33,
            "yanchor": "top",
            "pad": {"r": 10, "t": 36},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.33,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Through: ",
                "font": {"color": muted_white},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=14))


---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
  - A casino is calm because its edge is fixed: $\text{Casino Edge} = -\mathbb{E}[\text{Net Payout}]$. Players feel every swing; the house only needs sizing and survival so it never blows out. In the zero-sum simulation, player wealth drifts to ruin while casino wealth is the exact mirror of aggregate losses
  - Traders are not the house. Their edge is not a constant; it lives inside a policy $\pi(a_t \mid s_t; \theta)$ whose parameters encode beliefs, fear, and learned behavior. Loss aversion and related biases push decisions away from optimal risk-taking — same pressure as any profession where you have to step up to the plate
  - Handing trades to an algorithm,  = f_{\text{model}}(s_t; \phi)$, does not remove emotion. Model choice and parameterization are still human. A fixed mean-reversion rule can survive early regime shifts and still fail when the market trends against its stale assumptions
  - Every modeled problem is a specification-and-parameter problem: $\pi(a_t \mid s_t; \theta) \implies f_{\text{model}}(a_t \mid s_t; \phi(\theta))$. The real world is non-stationary, so you always need to update — the only question is when, and how aggressively
  - Bottom line: you cannot remove emotion from trading because someone is always steering the ship. Survival comes from qualitative Bayesian updating of the policy — daily work on when to trust the model, when to revise it, and how hard to turn the dial

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System


---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$